# RQ2 diagnostic alignment

Original notebook used to calculate skill-wise diagnostic correlations. Edit only the configuration paths before running.


## Path configuration

The analysis logic below is unchanged from the research notebook. Theta CSVs
are generated model outputs, not raw input data: train the selected model and
run its CP2 and CP4 diagnosis commands first. Then replace every
`REPLACE_WITH_...` value with the named generated output or authorized target
file. Paths may be absolute or repository-root-relative. Analysis outputs are
written under the repository's `outputs/` directory.


In [ ]:
# =========================================
# Cell 1: Import + Path Settings
# =========================================

import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import display
from pathlib import Path

# USER CONFIGURATION: replace these six input paths before running.
THETA_CP2_PATH = Path("REPLACE_WITH_CP2_THETA_CSV")
THETA_CP4_PATH = Path("REPLACE_WITH_CP4_THETA_CSV")
TARGET_MID_PATH = Path("REPLACE_WITH_MID_SCORE_CSV")
TARGET_FINAL_PATH = Path("REPLACE_WITH_FINAL_SCORE_CSV")
Q_MID_PATH = Path("REPLACE_WITH_MID_Q_MATRIX_CSV")
Q_FINAL_PATH = Path("REPLACE_WITH_FINAL_Q_MATRIX_CSV")

# This output path normally does not need to be changed.
OUTPUT_DIR = Path("outputs/RQ2_Diagnostic_Alignment/full_model/corr_theta_explicit_S1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MID_ID_MIN, MID_ID_MAX = 115, 229
FINAL_ID_MIN, FINAL_ID_MAX = 345, 459

MID_ITEMS_ALL = [8, 9, 10, 11]
FINAL_ITEMS_ALL = [19, 20, 21, 22]

SPECIFIC_SCORE_MODE = "mean"

MIN_N_PER_SKILL = 3

print("Configuration loaded")
print("THETA_CP2_PATH  =", THETA_CP2_PATH)
print("THETA_CP4_PATH  =", THETA_CP4_PATH)
print("TARGET_MID_PATH =", TARGET_MID_PATH)
print("TARGET_FINAL_PATH =", TARGET_FINAL_PATH)
print("Q_MID_PATH      =", Q_MID_PATH)
print("Q_FINAL_PATH    =", Q_FINAL_PATH)
print("OUTPUT_DIR      =", OUTPUT_DIR)

all_paths = [
    THETA_CP2_PATH, THETA_CP4_PATH,
    TARGET_MID_PATH, TARGET_FINAL_PATH,
    Q_MID_PATH, Q_FINAL_PATH
]

print("\n===== CHECK FILE EXISTS =====")
for p in all_paths:
    print(f"{p.name:<25} ->", p.exists())

#   theta CP2, theta CP4, score Mid, score Final, q_mid, q_final


In [ ]:
# =========================================
# Cell 2: Load Theta Files (CP2 / CP4) and Convert to Long Format
#     CP2 theta -> Mid
#     CP4 theta -> Final
# =========================================

def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

def load_theta_file(path, target_exam_name):
    df = pd.read_csv(path)
    df = clean_columns(df)

    if "user_id" in df.columns:
        student_col = "user_id"
    elif "student_id" in df.columns:
        student_col = "student_id"
    else:
        raise ValueError(
            f"Neither user_id nor student_id was found in {path}\n"
            f"available columns: {df.columns.tolist()}"
        )

    skill_cols = [c for c in df.columns if str(c).startswith("K")]
    if len(skill_cols) == 0:
        raise ValueError(
            f"No skill columns such as K0, K1, ... were found in {path}"
        )

    long_df = df[[student_col] + skill_cols].copy()
    long_df = long_df.rename(columns={student_col: "student_id"})

    long_df["student_id"] = pd.to_numeric(long_df["student_id"], errors="coerce")
    long_df = long_df.dropna(subset=["student_id"]).copy()
    long_df["student_id"] = long_df["student_id"].astype(int)

    long_df = long_df.melt(
        id_vars=["student_id"],
        value_vars=skill_cols,
        var_name="skill",
        value_name="theta"
    )

    long_df["skill"] = (
        long_df["skill"]
        .astype(str)
        .str.replace("K", "", regex=False)
        .str.strip()
    )
    long_df["skill"] = pd.to_numeric(long_df["skill"], errors="coerce")
    long_df["theta"] = pd.to_numeric(long_df["theta"], errors="coerce")

    long_df = long_df.dropna(subset=["student_id", "skill", "theta"]).copy()
    long_df["skill"] = long_df["skill"].astype(int)
    long_df["target_exam"] = target_exam_name

    return long_df[["student_id", "target_exam", "skill", "theta"]].sort_values(
        ["target_exam", "skill", "student_id"]
    ).reset_index(drop=True)

theta_cp2_df = load_theta_file(THETA_CP2_PATH, "Mid")
theta_cp4_df = load_theta_file(THETA_CP4_PATH, "Final")

print("theta_cp2_df shape =", theta_cp2_df.shape)
print("theta_cp4_df shape =", theta_cp4_df.shape)

display(theta_cp2_df.head(10))
display(theta_cp4_df.head(10))



In [ ]:
# =========================================
# Cell 3: Load Score Files (Mid / Final) and Standardize to Wide Format
# =========================================

def find_student_col(df):
    candidates = ["user_id", "student_id", "cpstudent", "student", "id", "เด็ก", "เด็ก/ข้อ"]
    for c in candidates:
        if c in df.columns:
            return c
    return None

def standardize_score_file(path, target_exam_name, valid_items):
    df = pd.read_csv(path)
    df = clean_columns(df)

    student_col = find_student_col(df)
    if student_col is None:
        raise ValueError(
            f"Student column not found in {path}\n"
            f"available columns: {df.columns.tolist()}"
        )

    if ("item_id" in df.columns) and ("score" in df.columns):
        long_df = df[[student_col, "item_id", "score"]].copy()
        long_df = long_df.rename(columns={student_col: "student_id"})
        long_df["student_id"] = pd.to_numeric(long_df["student_id"], errors="coerce")
        long_df["item_id"] = pd.to_numeric(long_df["item_id"], errors="coerce")
        long_df["score"] = pd.to_numeric(long_df["score"], errors="coerce")

        long_df = long_df.dropna(subset=["student_id", "item_id", "score"]).copy()
        long_df["student_id"] = long_df["student_id"].astype(int)
        long_df["item_id"] = long_df["item_id"].astype(int)

        long_df = long_df[long_df["item_id"].isin(valid_items)].copy()

        wide_df = long_df.pivot_table(
            index="student_id",
            columns="item_id",
            values="score",
            aggfunc="first"
        ).reset_index()

        wide_df.columns = ["student_id"] + [str(c) for c in wide_df.columns.tolist()[1:]]

    else:
        wide_df = df.copy()
        wide_df = wide_df.rename(columns={student_col: "student_id"})
        wide_df["student_id"] = pd.to_numeric(wide_df["student_id"], errors="coerce")
        wide_df = wide_df.dropna(subset=["student_id"]).copy()
        wide_df["student_id"] = wide_df["student_id"].astype(int)

        keep_cols = ["student_id"] + [str(i) for i in valid_items if str(i) in wide_df.columns]
        wide_df = wide_df[keep_cols].copy()

        for c in wide_df.columns:
            if c != "student_id":
                wide_df[c] = pd.to_numeric(wide_df[c], errors="coerce")

    wide_df["target_exam"] = target_exam_name

    item_cols = [str(i) for i in valid_items if str(i) in wide_df.columns]
    wide_df = wide_df[["student_id", "target_exam"] + item_cols].copy()

    return wide_df.sort_values("student_id").reset_index(drop=True)

score_mid_df = standardize_score_file(TARGET_MID_PATH, "Mid", MID_ITEMS_ALL)
score_final_df = standardize_score_file(TARGET_FINAL_PATH, "Final", FINAL_ITEMS_ALL)

print("score_mid_df shape =", score_mid_df.shape)
print("score_final_df shape =", score_final_df.shape)

display(score_mid_df.head())
display(score_final_df.head())



In [ ]:
# =========================================
# Cell 4: Concatenate Theta + Concatenate Scores
# =========================================

theta_all_df = pd.concat([theta_cp2_df, theta_cp4_df], axis=0, ignore_index=True)
score_all_df = pd.concat([score_mid_df, score_final_df], axis=0, ignore_index=True)

print("theta_all_df shape =", theta_all_df.shape)
print("score_all_df shape =", score_all_df.shape)

print("\nTarget-exam counts in theta_all_df")
print(theta_all_df["target_exam"].value_counts(dropna=False))

print("\nTarget-exam counts in score_all_df")
print(score_all_df["target_exam"].value_counts(dropna=False))

display(theta_all_df)
display(score_all_df)



In [ ]:
# =========================================
# Cell 5: Merge Theta with Scores
# =========================================

dup_check = score_all_df.duplicated(subset=["student_id", "target_exam"]).sum()
print("Duplicate rows in score_all_df by (student_id, target_exam) =", dup_check)

if dup_check > 0:
    raise ValueError("score_all_df must have one row per student_id and target_exam")

pooled_df = theta_all_df.merge(
    score_all_df,
    on=["student_id", "target_exam"],
    how="left",
    validate="m:1"
).copy()

print("pooled_df shape =", pooled_df.shape)
display(pooled_df)



In [ ]:
# =========================================
# Cell 6: Load Q-Matrix and Build Skill-to-Items Maps
# =========================================

def load_q_matrix_and_build_map(csv_path: str):
    q_df = pd.read_csv(csv_path)
    q_df = clean_columns(q_df)

    if "item_id" not in q_df.columns:
        raise ValueError(f"{csv_path} must contain item_id")

    q_df["item_id"] = pd.to_numeric(q_df["item_id"], errors="coerce")
    q_df = q_df.dropna(subset=["item_id"]).copy()
    q_df["item_id"] = q_df["item_id"].astype(int)

    skill_cols = [c for c in q_df.columns if str(c).startswith("K")]
    if len(skill_cols) == 0:
        raise ValueError(f"No K0, K1, ... columns were found in {csv_path}")

    for c in skill_cols:
        q_df[c] = pd.to_numeric(q_df[c], errors="coerce").fillna(0)

    skill_to_items = {}
    for c in skill_cols:
        skill_idx = int(str(c).replace("K", ""))
        related_items = q_df.loc[q_df[c] > 0, "item_id"].astype(int).tolist()
        skill_to_items[skill_idx] = related_items

    return q_df, skill_to_items

q_mid_df, skill_to_mid_items = load_q_matrix_and_build_map(Q_MID_PATH)
q_final_df, skill_to_final_items = load_q_matrix_and_build_map(Q_FINAL_PATH)

print("Example Mid skill_to_items")
for k in list(skill_to_mid_items.keys()):
    print(f"skill {k}: {skill_to_mid_items[k]}")

print("\nExample Final skill_to_items")
for k in list(skill_to_final_items.keys()):
    print(f"skill {k}: {skill_to_final_items[k]}")

display(q_mid_df)
display(q_final_df)



In [ ]:
# =========================================
# Cell 7: Compute Specific Score
# =========================================

def aggregate_values(values: pd.Series, mode: str = "sum"):
    values = pd.to_numeric(values, errors="coerce").dropna()
    if len(values) == 0:
        return np.nan
    if mode == "sum":
        return values.sum()
    elif mode == "mean":
        return values.mean()
    else:
        raise ValueError("mode must be 'sum' or 'mean'")

def get_related_items(skill, target_exam):
    skill = int(skill)
    if target_exam == "Mid":
        return skill_to_mid_items.get(skill, [])
    elif target_exam == "Final":
        return skill_to_final_items.get(skill, [])
    else:
        return []

def compute_specific_score(row):
    related_items = get_related_items(row["skill"], row["target_exam"])
    related_cols = [str(i) for i in related_items if str(i) in pooled_df.columns]

    if len(related_cols) == 0:
        return np.nan

    return aggregate_values(row[related_cols], mode=SPECIFIC_SCORE_MODE)

def related_items_as_text(row):
    related_items = get_related_items(row["skill"], row["target_exam"])
    return ",".join(map(str, related_items))

pooled_df["specific_score"] = pooled_df.apply(compute_specific_score, axis=1)
pooled_df["related_items_used"] = pooled_df.apply(related_items_as_text, axis=1)

analysis_df = pooled_df[[
    "student_id", "target_exam", "skill", "theta", "specific_score", "related_items_used"
]].copy()

print("analysis_df shape =", analysis_df.shape)
display(analysis_df)



In [ ]:
# =========================================
# Cell 8: Debug One Skill
# =========================================

SKILL_TO_DEBUG = 6

debug_cols = ["student_id", "target_exam", "skill", "theta", "specific_score", "related_items_used"] + \
             [c for c in [str(i) for i in MID_ITEMS_ALL + FINAL_ITEMS_ALL] if c in pooled_df.columns]

debug_df = pooled_df.loc[pooled_df["skill"] == SKILL_TO_DEBUG, debug_cols].copy()
debug_df = debug_df.sort_values(["target_exam", "student_id"]).reset_index(drop=True)

print(f"DEBUG skill = {SKILL_TO_DEBUG}")
display(debug_df)



In [ ]:
# =========================================
# Cell 9: Compute Pooled Spearman by Skill
# =========================================

results = []

for skill, g in analysis_df.groupby("skill"):
    g = g[["student_id", "target_exam", "skill", "theta", "specific_score"]].dropna().copy()

    n = len(g)
    theta_unique = g["theta"].nunique()
    score_unique = g["specific_score"].nunique()

    if n < MIN_N_PER_SKILL:
        rho, p_value, status = np.nan, np.nan, f"skip: n<{MIN_N_PER_SKILL}"
    elif theta_unique < 2:
        rho, p_value, status = np.nan, np.nan, "skip: theta constant"
    elif score_unique < 2:
        rho, p_value, status = np.nan, np.nan, "skip: specific_score constant"
    else:
        rho, p_value = spearmanr(g["theta"], g["specific_score"])
        status = "ok"

    results.append({
        "skill": int(skill),
        "n_students": n,
        "theta_unique": theta_unique,
        "specific_score_unique": score_unique,
        "spearman_rho": rho,
        "p_value": p_value,
        "status": status
    })

results_df = pd.DataFrame(results).sort_values("skill").reset_index(drop=True)

display(results_df)



In [ ]:
# =========================================
# Cell 10: Summary + Save Output
# =========================================

valid_results_df = results_df.loc[
    results_df["status"].eq("ok") & results_df["spearman_rho"].notna()
].copy()

summary_df = pd.DataFrame([{
    "specific_score_mode": SPECIFIC_SCORE_MODE,
    "n_skills_total": len(results_df),
    "n_skills_valid": len(valid_results_df),
    "mean_spearman": valid_results_df["spearman_rho"].mean() if len(valid_results_df) else np.nan,
    "median_spearman": valid_results_df["spearman_rho"].median() if len(valid_results_df) else np.nan,
    "min_spearman": valid_results_df["spearman_rho"].min() if len(valid_results_df) else np.nan,
    "max_spearman": valid_results_df["spearman_rho"].max() if len(valid_results_df) else np.nan,
}])

display(summary_df)

analysis_out = OUTPUT_DIR / "pooled_theta_specific_score_table.csv"
results_out = OUTPUT_DIR / "pooled_spearman_by_skill.csv"
summary_out = OUTPUT_DIR / "pooled_spearman_summary.csv"

analysis_df.to_csv(analysis_out, index=False, encoding="utf-8-sig")
results_df.to_csv(results_out, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_out, index=False, encoding="utf-8-sig")

print("saved:", analysis_out)
print("saved:", results_out)
print("saved:", summary_out)

